# Sentinel-1 Float with Gamma MAP

This tutorial captures the SAR feature workflow used for biomass-style remote sensing features: linear Sentinel-1, optional Gamma MAP speckle filtering, VH/VV ratio, then temporal reduction.


In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT.name != "mia-tutorials" and ROOT.parent != ROOT:
    ROOT = ROOT.parent

DATA = ROOT / "data"
REPOS = ROOT.parent
ROOT

from geecomposer import compose, initialize
from geecomposer.datasets.sentinel1_preprocessing import gamma_map
from geecomposer.transforms.expressions import expression_transform

AOI = DATA / "geecomposer" / "sample_aoi.geojson"


In [ ]:
# Local validation from the package tests: kernel size must be a positive odd integer.
for k in [1, 5, 7]:
    fn = gamma_map(kernel_size=k)
    print(fn.__name__)


In [ ]:
vh_vv = expression_transform(
    expression="vh / vv",
    band_map={"vh": "VH", "vv": "VV"},
    name="vh_vv_ratio",
)

compose_kwargs = dict(
    dataset="sentinel1_float",
    aoi=str(AOI),
    start="2025-01-01",
    end="2025-12-31",
    preprocess=gamma_map(kernel_size=7),
    transform=vh_vv,
    reducer="median",
    filters={"instrumentMode": "IW", "polarizations": ["VV", "VH"]},
)
compose_kwargs


In [ ]:
RUN_EE = False

if RUN_EE:
    initialize(project="manglariars", authenticate=False)
    img = compose(**compose_kwargs)
    print("bands:", img.bandNames().getInfo())
else:
    print("Skipped live compose. Set RUN_EE = True after Earth Engine credentials are ready.")
